## Cell 1 - Imports

In [ ]:
# Import the libraries used in this notebook section.
import pandas as pd
import numpy as np

: 

## Cell 2 - Load the TED dataset

In [ ]:
# Set up project paths so files can be read and outputs can be organised consistently.
file_path = "../data/raw/ted_most_recent_50000.csv"

# Load the dataset needed for the next analysis step.
df = pd.read_csv(file_path, low_memory=False)

# Inspect the data to confirm the structure and values look reasonable.
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

## Cell 3 - Inspect columns and missing values

In [ ]:
print("Columns:")
# Inspect the data to confirm the structure and values look reasonable.
print(df.columns.tolist())

null_summary = pd.DataFrame({
    # Handle missing values so later transformations and models do not fail.
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().sum() / len(df)) * 100,
    "nunique": df.nunique(dropna=False)
}).sort_values("null_pct", ascending=False)

display(null_summary)

## Cell 4 - Remove exact duplicate rows

In [ ]:
# Filter the data to keep the records relevant for this step.
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]

print(f"Rows before duplicate removal: {before}")
print(f"Rows after duplicate removal:  {after}")
print(f"Exact duplicates removed:      {before - after}")

## Cell 5 - Choose the target

In [ ]:
TARGET = "VALUE_EURO"
print("Chosen target:", TARGET)

## Cell 6 - Drop rows where target is missing or invalid

In [ ]:
# Convert target to numeric if needed
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

before = len(df)

# Remove missing target rows
df = df[df[TARGET].notna()]

# Remove zero or negative values if present
df = df[df[TARGET] > 0]

after = len(df)

print(f"Rows before target filtering: {before}")
print(f"Rows after target filtering:  {after}")
print(f"Rows removed:                 {before - after}")

## Cell 7 - Drop fully null, near-null, and constant columns

In [ ]:
# Fully null columns
full_null_cols = [col for col in df.columns if df[col].isna().all()]

# Near-null columns (95%+ missing)
near_null_cols = [col for col in df.columns if df[col].isna().mean() >= 0.95]

# Constant columns
constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]

# Inspect the data to confirm the structure and values look reasonable.
print("Fully null columns:", full_null_cols)
print("Near-null columns:", near_null_cols)
print("Constant columns:", constant_cols)

drop_stage1 = sorted(set(full_null_cols + near_null_cols + constant_cols))
print("\nColumns to drop in stage 1:", drop_stage1)

df = df.drop(columns=drop_stage1, errors="ignore")

print("Shape after stage 1 drop:", df.shape)

## Cell 8 - Drop redundant columns

In [ ]:
# Filter the data to keep the records relevant for this step.
redundant_cols = []

# Common TED redundancies seen earlier
for col in ["source_year", "VALUE_EURO_FIN_2"]:
    if col in df.columns:
        redundant_cols.append(col)

# Inspect the data to confirm the structure and values look reasonable.
print("Redundant columns to drop:", redundant_cols)

df = df.drop(columns=redundant_cols, errors="ignore")

print("Shape after redundant drop:", df.shape)

## Cell 9 - Drop irrelevant ID/admin columns

In [ ]:
id_admin_cols = [
    "ID_NOTICE_CAN",
    "TED_NOTICE_URL",
    "ID_TYPE",
    "XSD_VERSION",
    "ID_LOT",
    "ID_AWARD",
    "ID_LOT_AWARDED",
    "CONTRACT_NUMBER",
    "CAE_NATIONALID",
    "CAE_ADDRESS",
    "CAE_POSTAL_CODE"
]

# Filter the data to keep the records relevant for this step.
existing_id_admin_cols = [col for col in id_admin_cols if col in df.columns]
# Inspect the data to confirm the structure and values look reasonable.
print("Dropping ID/admin columns:", existing_id_admin_cols)

df = df.drop(columns=existing_id_admin_cols, errors="ignore")

print("Shape after ID/admin drop:", df.shape)

## Cell 10 - Drop award-stage / leakage columns

In [ ]:
leakage_cols = [
    "AWARD_VALUE_EURO",
    "AWARD_EST_VALUE_EURO",
    "AWARD_VALUE_EURO_FIN_1",
    "WIN_NAME",
    "WIN_NATIONALID",
    "WIN_ADDRESS",
    "WIN_TOWN",
    "WIN_POSTAL_CODE",
    "WIN_COUNTRY_CODE",
    "NUMBER_AWARDS",
    "B_AWARDED_TO_A_GROUP",
    "B_CONTRACTOR_SME",
    "B_SUBCONTRACTED",
    "DT_AWARD",
    "INFO_ON_NON_AWARD"
]

# Filter the data to keep the records relevant for this step.
existing_leakage_cols = [col for col in leakage_cols if col in df.columns]
# Inspect the data to confirm the structure and values look reasonable.
print("Dropping leakage columns:", existing_leakage_cols)

df = df.drop(columns=existing_leakage_cols, errors="ignore")

print("Shape after leakage drop:", df.shape)

## Cell 11 - Drop low-relevance procedural columns

In [ ]:
procedural_cols = [
    "CORRECTIONS",
    "B_MULTIPLE_CAE",
    "B_MULTIPLE_COUNTRY",
    "B_ON_BEHALF",
    "B_INVOLVES_JOINT_PROCUREMENT",
    "B_AWARDED_BY_CENTRAL_BODY",
    "B_GPA",
    "INFO_UNPUBLISHED",
    "OUT_OF_DIRECTIVES",
    "B_ELECTRONIC_AUCTION",
    "B_DYN_PURCH_SYST",
    "B_FRA_AGREEMENT",
    "FRA_ESTIMATED",
    "B_FRA_CONTRACT",
    "B_EU_FUNDS",
    "VALUE_EURO_FIN_1"   # drop if using VALUE_EURO as target
]

# Filter the data to keep the records relevant for this step.
existing_procedural_cols = [col for col in procedural_cols if col in df.columns]
# Inspect the data to confirm the structure and values look reasonable.
print("Dropping procedural / low-value columns:", existing_procedural_cols)

df = df.drop(columns=existing_procedural_cols, errors="ignore")

print("Shape after procedural drop:", df.shape)

## Cell 12 - Keep only the core modelling columns

In [ ]:
core_columns = [
    "TITLE",
    "CPV",
    "ADDITIONAL_CPVS",
    "TYPE_OF_CONTRACT",
    "TOP_TYPE",
    "MAIN_ACTIVITY",
    "CAE_TYPE",
    "ISO_COUNTRY_CODE",
    "TAL_LOCATION_NUTS",
    "LOTS_NUMBER",
    "DT_DISPATCH",
    TARGET
]

# Filter the data to keep the records relevant for this step.
existing_core_columns = [col for col in core_columns if col in df.columns]
# Define the model or model set that will be trained and compared.
df_model = df[existing_core_columns].copy()

# Inspect the data to confirm the structure and values look reasonable.
print("Final modelling columns:")
print(df_model.columns.tolist())

print("\nShape of modelling dataframe:", df_model.shape)
display(df_model.head())

## Cell 13 - Convert dates and engineer time features

In [ ]:
if "DT_DISPATCH" in df_model.columns:
    # First try day-first parsing, which is common in TED-style European dates
    df_model["DT_DISPATCH"] = pd.to_datetime(
        df_model["DT_DISPATCH"],
        errors="coerce",
        dayfirst=True
    )

    # Filter the data to keep the records relevant for this step.
    df_model["DISPATCH_YEAR"] = df_model["DT_DISPATCH"].dt.year
    df_model["DISPATCH_MONTH"] = df_model["DT_DISPATCH"].dt.month
    df_model["DISPATCH_QUARTER"] = df_model["DT_DISPATCH"].dt.quarter

    # Define the model or model set that will be trained and compared.
    df_model = df_model.drop(columns=["DT_DISPATCH"])

print("Columns after date engineering:")
# Inspect the data to confirm the structure and values look reasonable.
print(df_model.columns.tolist())

## Cell 14 - Handle missing values

In [ ]:
# Fill text/categorical columns with "Unknown"
categorical_cols = df_model.select_dtypes(include=["object", "string"]).columns.tolist()

for col in categorical_cols:
    # Handle missing values so later transformations and models do not fail.
    df_model[col] = df_model[col].fillna("Unknown")

# Fill numeric columns with median, except target
numeric_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
# Filter the data to keep the records relevant for this step.
numeric_feature_cols = [col for col in numeric_cols if col != TARGET and col != "LOG_" + TARGET]

# Choose the columns that will be used as features or summary variables.
for col in numeric_feature_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

print("Remaining missing values:")
# Inspect the data to confirm the structure and values look reasonable.
display(df_model.isna().sum().sort_values(ascending=False))

## Cell 15 - Log transform the target

In [ ]:
# Convert values into analysis-friendly numeric, date, or text formats.
df_model["LOG_" + TARGET] = np.log1p(df_model[TARGET])

# Inspect the data to confirm the structure and values look reasonable.
print(df_model[[TARGET, "LOG_" + TARGET]].head())

## Cell 16 - Final inspection

In [ ]:
# Inspect the data to confirm the structure and values look reasonable.
print("Final cleaned modelling dataset shape:", df_model.shape)
print("\nFinal columns:")
print(df_model.columns.tolist())

display(df_model.head())

## Cell 17 - Optional: save cleaned dataset

In [ ]:
output_file = "../data/processed/ted_cleaned_for_cost_prediction.csv"
# Save the processed output so later notebooks or report sections can reuse it.

df_model.to_csv(output_file, index=False)

print(f"Saved cleaned dataset to: {output_file}")